# Unavailable borrower size in SCR.data — business credit, Brazil

How much business credit is granted without company revenue on file, broken down
by type of lending institution, nationwide, from January 2019 to June 2026.

**Source:** SCR.data, Central Bank of Brazil (ODbL licence).
**Methodological decisions:** see `DECISIONS.md` in the repository root.

> **`carteira_ativa` is a stock, not a flow.** It is the balance outstanding at
> month end — every contract still being repaid, whenever it was signed. So *"3.1%
> of bank credit has no size on file"* reads: of every R$ 100 that companies owed
> banks on that date, R$ 3.10 sat in contracts whose borrower has no revenue on
> record. It says nothing about new lending that month. A share can move because
> new credit came in, or because existing contracts were re-registered, and this
> series cannot separate the two.

> **On "business credit".** The extract filters SCR.data to client type `PJ`,
> which the Central Bank renders in English as *legal entities* — every
> registered company, from a sole trader to a listed multinational. It is read
> here as **business credit**, never *corporate credit*: in European and US
> banking, "corporate" denotes the large-company segment, and using it would
> imply a size restriction this extract does not make. Borrower size is the
> variable under study, so it cannot also be smuggled into the scope.

Working notebook. Consolidated conclusions go to the README.

> **Scope note.** An earlier version of this notebook was restricted to one state
> (Paraná). The scope was widened to the whole country on 2026-08-20 and carried
> into the code on 2026-08-24. Every reading below is nationwide. Where a Paraná
> figure was superseded, the amendment is stated in place rather than removed.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import numpy as np
import pandas as pd

import style
style.apply()

# Data paths are anchored on the repository root rather than the working
# directory, which differs between `jupyter lab` and VS Code.
ROOT = Path.cwd()
while not (ROOT / 'src').is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / 'src').is_dir():
    raise RuntimeError(
        'repository root not found — open this notebook from inside the '
        f'SCR_Bacen working tree (current directory: {Path.cwd()})')
print(f'repository root: {ROOT}')

PROCESSED = ROOT / 'data' / 'processed'
PARQUET = PROCESSED / 'scrdata_pj_br.parquet'                  # nationwide
INGESTION_LOG = PROCESSED / 'scrdata_pj_br_ingestion_log.json'

# Column names are the Central Bank's, unchanged. Glossed here rather than
# renamed: anyone searching SCR.data for a name used in this repository finds it.
COLUMNS = {
    'data_base':            'reference month',
    'segmento':             'lender segment (regulatory classification)',
    'carteira_ativa':       'active portfolio, BRL',
    'porte_indisponivel':   'outstanding with borrower size not reported, BRL',
    'numero_de_operacoes':  'contract count, withheld where cells are small',
    'operacoes_suprimidas': 'flag: contract count withheld for confidentiality',
}

# Segment values are the data itself. See the glossary in README.md.
STUDIED = ['Banco', 'Fintech', 'Instituição de pagamento']

## 1. Load and sanity-check the extract

Before any analysis: is the period complete, and does the volume match what the
ingestion routine reported?

In [ ]:
# Only the columns the analysis uses: the nationwide extract is ~10.3M rows and
# the full column set does not fit comfortably in memory.
df = pd.read_parquet(PARQUET, columns=list(COLUMNS))

print(f'rows     : {len(df):,}')
print(
    f'period   : {df["data_base"].min():%m-%Y} to {df["data_base"].max():%m-%Y}')
print(f'months   : {df["data_base"].nunique()}')
print(f'segments : {df["segmento"].nunique()}')

### 1.1 Cross-check against the ingestion log

The ingestion log is the only versioned data artefact in this repository. It
records, per monthly file, how many rows the source held and how many survived
the business-borrower filter. Two uses here.

**First, an integrity check.** If the Parquet and the log disagree on row count
or period, one of them is stale and nothing below can be trusted.

**Second, a break detector.** A month where the source file itself changes shape
is a month where any series built on it deserves suspicion. This is cheap to
compute and it does not depend on the analysis being right.

In [ ]:
log = json.loads(INGESTION_LOG.read_text(encoding='utf-8'))

print(f'log rows kept    : {log["rows_kept"]:,}   (parquet: {len(df):,})')
print(f'log rows read    : {log["rows_read"]:,}')
print(f'log months       : {len(log["months"])}')
print(f'suppression rate : {log["suppression_rate"]:.2%}')
print(
    f'size unavailable : {log["porte_indisponivel_rate"]:.2%}  (share of rows, unweighted)')

ingestion = (pd.DataFrame(log['months'])
               .assign(month=lambda t: pd.to_datetime(
                   t['file'].str[8:14], format='%Y%m').dt.to_period('M')))

ingestion['read_delta_%'] = ingestion['rows_read'].pct_change() * 100
ingestion['kept_delta_%'] = ingestion['rows_kept'].pct_change() * 100
# Share of the published file that is business credit. A step here is a change
# in the composition of the source, not in the credit market.
ingestion['pj_share_%'] = ingestion['rows_kept'] / ingestion['rows_read'] * 100
ingestion['pj_share_delta_pp'] = ingestion['pj_share_%'].diff()

(ingestion.set_index('month')[['read_delta_%', 'kept_delta_%',
                               'pj_share_%', 'pj_share_delta_pp']]
 .reindex(ingestion['pj_share_delta_pp'].abs().nlargest(6).index
          .map(ingestion['month'].get))
 .round(2))

### Reading

Two months stand out, and they are **not the same kind of event**.

| | rows read | rows kept (business) | business share of file |
|---|---|---|---|
| June 2024 | −2.19% | −2.13% | no step |
| July 2025 | **−4.58%** | −1.02% | **+1.49 pp, permanent** |

**June 2024 is proportional.** The whole file contracts and the business slice
contracts with it, by the same amount. Nothing about the composition of the
source changed. Whatever moved that month moved everything.

**July 2025 is not.** The file loses 4.6% of its rows while the business slice
loses only 1%. What disappeared was disproportionately *not* business credit. The
business share of the file steps up by 1.5 pp and **does not revert**: the
annual mean goes from 39.4% in 2023 to 40.5% in 2025 and 40.9% in 2026.

**A third month, larger than both.** March 2019 shifts the composition by
−1.56 pp, more than July 2025. It sits in the first quarter of the series, where
the file was still growing month on month, and it moves in the opposite
direction. It is flagged here rather than explained: the fintech series does not
begin until July 2019, so nothing in the finding rests on it.

These are structural changes in what the Central Bank publishes, dated to
specific months, measured from the audit trail rather than inferred from the
result. Section 7 returns to July 2025.

## 2. Who is who in Brazilian business credit

Before comparing proportions across segments, we need to know how big each one
is. A proportion without volume beside it is misleading.

In [ ]:
monthly_total = df.groupby('data_base', observed=True)['carteira_ativa'].sum()

by_segment = (df.groupby(['data_base', 'segmento'], observed=True)['carteira_ativa']
                .sum().unstack('segmento'))

market_share = by_segment.div(monthly_total, axis=0)

# Outstanding is in BRL. Nationwide the bank median runs past 1e12, and pandas
# renders that in scientific notation — unreadable beside a percentage column.
# Billions keep all three segments legible without collapsing the small ones:
# in trillions the fintech row would round to zero.
BILLION = 1e9

summary = pd.DataFrame({
    'median_outstanding_BRL_bn': (by_segment.median() / BILLION).round(1),
    'median_share_%': (market_share.median() * 100).round(3),
    'max_share_%': (market_share.max() * 100).round(3),
}).sort_values('median_share_%', ascending=False)

# Nominal BRL, not deflated: the median spans January 2019 to June 2026. It is
# used here to order magnitudes, not to compare purchasing power across years.
#
# Formatting a copy rather than `summary.style`: the Styler accessor pulls in
# jinja2, and a thousands separator is not worth a seventh dependency. `summary`
# itself stays numeric and sortable.
summary.assign(**{'median_outstanding_BRL_bn':
                  summary['median_outstanding_BRL_bn'].map('{:,.1f}'.format)})

### Reading

SCR.data splits lenders into eight segments. Banks hold roughly three quarters of
business credit (76.7% median share), followed by development banks (14.9%) and
credit unions (6.3%). The three segments this notebook studies sit at opposite
ends of that range: `Banco` at 76.7%, `Fintech` at 0.047%, `Instituição de
pagamento` at 0.026%.

**Consequence for the thesis:** any high proportion found in the latter two is a
large slice of a small whole. Widening the scope from one state to the country
multiplied the denominator roughly fifteenfold, which is what makes the fintech
series worth reading at all — but it did not make the segment large. That has to
be stated alongside every figure below, not instead of them.

**A scope question this table raises.** Two segments left out of the study are far
larger than two segments kept in. `Cooperativa` and `Desenvolvimento/Fomento`
together hold about 21% of the market, and a credit union underwrites against
declared revenue much as a bank does — so the thesis predicts they should behave
like the control, not like fintechs. Section 3 tests exactly that, on all eight
segments, before the series narrows to three.

**On the labels.** Segment values stay in Portuguese, as the Central Bank
publishes them. They are data, not vocabulary: rewriting them would break the
match between this repository and the source. `README.md` carries a glossary of
what each one covers.

## 3. The decisive test — unavailable size within each segment

The original hypothesis was that traditional banks cannot classify a meaningful
share of small companies. If that were true, the share of unavailable size
*within* the Bank segment would have to be high.

Measured on `outstanding` — see `DECISIONS.md` for why.

In [ ]:
latest = df[df['data_base'] == df['data_base'].max()]

test = (latest.groupby(['segmento', 'porte_indisponivel'], observed=True)['carteira_ativa']
              .sum().unstack('porte_indisponivel', fill_value=0))
test.columns = ['size_reported', 'size_unavailable']
test['unavailable_share_%'] = (
    test['size_unavailable'] / test.sum(axis=1) * 100).round(1)

# Same billions conversion as section 2, and for the same reason: summed
# nationwide these two columns run past 1e12 and print in scientific notation.
# The share column is the one the reading uses; the value columns are here to
# show where the ratio comes from, so one decimal is enough — segments below
# 100 million round to 0.0 and that is acceptable at this resolution.
#
# These three names are outputs of the analysis, not fields of the source, so
# they are in English. Source field names are never renamed; derived ones are
# named by the analyst.
test[['size_reported', 'size_unavailable']] = (
    test[['size_reported', 'size_unavailable']] / BILLION).round(1)
test.columns = ['size_reported_BRL_bn', 'size_unavailable_BRL_bn',
                'unavailable_share_%']

test.sort_values('unavailable_share_%', ascending=False)

### Reading

The original hypothesis **does not hold**: banks know the revenue of almost
everything they lend to. In June 2026, nationwide, only 3.1% of bank business
credit carries no borrower size.

What is more useful is that the eight segments do not scatter. They order:

| Segment | No size on file | What it underwrites against |
|---|---|---|
| `Fintech` | 40.3% | transaction flow |
| `Instituição de pagamento` | 8.0% | transaction flow |
| `Financeira` | 7.4% | mixed |
| `Desenvolvimento/Fomento` | 6.4% | declared revenue |
| `Cooperativa` | 3.3% | declared revenue |
| `Banco` | 3.1% | declared revenue |
| `Arrendamento` | 3.1% | the asset itself |
| `Outros` | 0.5% | — |

**This is the strongest evidence in the notebook, and it was not designed.** The
ranking is not by size, age or regulatory tier: `Desenvolvimento/Fomento` is a
hundred times larger than `Fintech` and sits near the bottom; `Cooperativa` holds
6.3% of the market and lands within 0.2 pp of the banks. What the ranking tracks
is how the lender decides. Every segment that underwrites against declared
revenue clusters between 3% and 6%. The two that underwrite against transaction
flow sit above 8%, and the purest case of it sits at 40%.

So the empty field describes **the method of whoever granted the credit**, not the
borrower. A bank underwrites against declared revenue, so the field is populated.
A lender underwriting against card receivables, instant payments and invoice
history has no reason to capture revenue at all. The question becomes how much
credit is granted without looking at revenue, and how fast that is growing.

**It also answers the scope question from section 2.** Credit unions and
development banks behave like banks, which is what the thesis predicts and what
makes narrowing to three defensible: `Banco` already represents that whole family,
and it represents it with the largest denominator of the eight. Section 8 keeps
the wider test on the list anyway — one month is one month.

> **Amended from the Paraná reading.** The state figures were 1% / 11% / 31% for
> bank, payment institution and fintech. The national figures are 3.1% / 8.0% /
> 40.3%. The ordering of bank against fintech survives and the gap widens; the
> payment-institution figure moves the other way. The old numbers are not
> relabelled — they are replaced, and the replacement is dated.

## 4. The monthly series, unfiltered

The raw chart first, with everything in it. It shows both the finding and the
problem.

In [ ]:
base = df[df['segmento'].isin(STUDIED)]

# Denominator: all active credit per month and segment.
total = base.groupby(['data_base', 'segmento'], observed=True)[
    'carteira_ativa'].sum()

# Numerator: the slice with no reported borrower size.
# reindex + fill_value=0 matters: a month with no such credit is a true zero,
# not a gap. Without it the line breaks and reads as missing data.
unavailable = (base[base['porte_indisponivel']]
               .groupby(['data_base', 'segmento'], observed=True)['carteira_ativa'].sum()
               .reindex(total.index, fill_value=0))

series = (unavailable / total).unstack('segmento')

# Both series charts share this ceiling. The filtered chart in section 5 drops
# exactly the extreme months, so letting matplotlib pick its own limits there
# would give two same-titled figures different rulers — and the filtered line
# would look flattened by the criterion rather than by the axis.
YMAX = series.max().max() * 1.05

fig, ax = style.figure('Unavailable borrower size as a share of outstanding credit',
                       'Share of outstanding',
                       subtitle='Business credit, Brazil — raw series, no stability filter')
series.plot(ax=ax, color=style.colors(series.columns))
# Dashes the orange line: it is 1.13:1 from the cyan one in luminance,
# so colour alone does not separate them in greyscale or in print.
style.dash(ax)
ax.set_ylim(0, YMAX)
# The series holds fractions; the narrative and README speak in per cent.
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1, decimals=0))
style.mark(ax, pd.Timestamp('2025-07-31'),
           'Jul 2025 — unexplained movement (see DECISIONS.md)')
style.finalize(ax, 'Source: SCR.data, Central Bank of Brazil (ODbL)')
plt.show()

### Reading

Two lines behave and one does not.

- **`Banco`**: moves far more than a first look suggests. It climbs through 2019
  to **14.7% (Dec 2019)**, collapses to about **1.5% during 2020**, and then
  drifts up again: 4.4% (Dec 2023), 3.1% (Jun 2026). Roughly a factor of ten
  between the low and the high.
- **`Fintech`**: starts near zero and rises steadily. A trend, not noise.
- **`Instituição de pagamento`**: 0.1% (Mar 2021) → 67.3% (Dec 2023) → 8.0% (Jun 2026).
  Spike and collapse. That is not economic behaviour.

> **Amendment — the control group, twice.** The Paraná version of this notebook
> read the bank line as flat at around 2% for seven years and called it *"close to
> a perfect control group"*. The first amendment, written before the series was
> plotted nationwide, put the range at 1.5%–5.2% and called it a factor of three.
> **That was also wrong, and in the same direction.** Section 4.1 surfaced
> Dec 2019 at 14.7%: the true range is roughly 1.4% to 14.7%, a factor of ten.
>
> The lesson repeats itself. Both errors came from describing a line from summary
> figures instead of from the series. The second one was written by someone who
> had already been caught by the first.
>
> **What survives.** Two things, and they are weaker than what was claimed before.
> First, over the last four years — the stretch where the fintech series is dense
> and passes the stability criterion — the bank sits between roughly 1.5% and 4.7%
> while the fintech travels from 5% to 40%. Second, the bank's own large movement
> is dated to 2019–2020 and runs in the *opposite* direction to the fintech rise
> that begins in 2020. So the bank is not an immobile control. It is a segment
> whose movement has a different shape, a different period and a different sign.
> That is still a contrast, and it is one that a collection-wide change could not
> produce.
>
> **What is now open.** Nobody has explained the 2019–2020 collapse in the bank
> line. It is listed in section 8.

## 4.1 The December 2023 peak in the bank line

The bank series reaches 4.4% in December 2023, its highest point since early
2019. The cause cannot be established from this base — that would need each
institution's internal credit policy. What *can* be done is to narrow the space of
explanations until only a named family of causes is left, using nothing but the
data already loaded.

Three checks, in order of how cheaply they kill a hypothesis.

In [ ]:
# Check 1 — numerator or denominator?
#
# A share rises two ways: credit without size came IN, or credit with size went
# OUT. If the bank's total portfolio shrank that month, the peak is arithmetic
# and has nothing to do with registration practice.

bank = base[base['segmento'] == 'Banco']

bank_total = bank.groupby('data_base', observed=True)['carteira_ativa'].sum()
bank_unavailable = (bank[bank['porte_indisponivel']]
                    .groupby('data_base', observed=True)['carteira_ativa'].sum()
                    .reindex(bank_total.index, fill_value=0))

window = slice('2023-06-30', '2024-06-30')
check1 = pd.DataFrame({
    'total_BRL_bn': (bank_total / BILLION).round(1),
    'unavailable_BRL_bn': (bank_unavailable / BILLION).round(1),
    'share_%': (bank_unavailable / bank_total * 100).round(2),
}).loc[window]
check1['total_mom_%'] = (bank_total.loc[window].pct_change() * 100).round(2)
check1['unavailable_mom_%'] = (
    bank_unavailable.loc[window].pct_change() * 100).round(2)
check1

In [ ]:
# Check 2 — is December systematically high, or was 2023 an exception?
#
# The seasonal hypothesis (year-end target pressure loosening registration
# discipline) makes a testable prediction: December should sit above its
# neighbours in most years, not just once.

bank_share = (bank_unavailable / bank_total * 100)

seasonal = pd.DataFrame({
    'nov_%': bank_share[bank_share.index.month == 11].values[:7],
    'dec_%': bank_share[bank_share.index.month == 12].values[:7],
    'jan_next_%': bank_share[bank_share.index.month == 1].values[1:8],
}, index=range(2019, 2026)).round(2)
seasonal['dec_minus_nov_pp'] = (seasonal['dec_%'] - seasonal['nov_%']).round(2)
seasonal

In [ ]:
# Check 3 — spread across the book, or concentrated in one product?
#
# `modalidade` is not in the six columns the notebook loads. Reading it for the
# whole series would not fit in memory, so this reads a narrow slice straight
# from the Parquet: banks only, one year around the peak. Row-group filters are
# applied before anything reaches pandas.

PEAK = pd.Timestamp('2023-12-31')

slice_cols = ['data_base', 'modalidade',
              'carteira_ativa', 'porte_indisponivel']
peak_slice = pd.read_parquet(
    PARQUET,
    columns=slice_cols,
    filters=[('segmento', '==', 'Banco'),
             ('data_base', '>=', pd.Timestamp('2023-11-30')),
             ('data_base', '<=', PEAK)],
)

by_product = (peak_slice.groupby(['data_base', 'modalidade'], observed=True)
                        .apply(lambda g: pd.Series({
                            'total': g['carteira_ativa'].sum(),
                            'unavailable': g.loc[g['porte_indisponivel'],
                                                 'carteira_ativa'].sum()}),
                               include_groups=False))
by_product['share_%'] = (by_product['unavailable'] / by_product['total'] * 100)

movement = (by_product['share_%'].unstack('data_base')
            .rename(columns=lambda c: f'{c:%Y-%m}'))
movement.columns = ['nov_%', 'dec_%']
movement['delta_pp'] = movement['dec_%'] - movement['nov_%']
# Weight by how much of the book each product carries: a 30 pp move in a
# product holding 0.1% of the portfolio explains nothing.
movement['dec_weight_%'] = (by_product['total'].unstack('data_base').iloc[:, -1]
                            / by_product['total'].unstack('data_base').iloc[:, -1].sum()
                            * 100)
movement.round(2).sort_values('delta_pp', ascending=False).head(10)

### Reading

**Check 1 — not a denominator effect.** The bank's book *grew* that month, from
R$ 1,774.7 bn to R$ 1,834.8 bn (+3.4%), while the balance with no size on file
grew from R$ 61.6 bn to R$ 80.1 bn — **+30.0% in a single month**. The share moved
+0.89 pp because the numerator jumped, not because the denominator shrank.

**Check 2 — there is a seasonal pattern, and it is not subtle.** December sits
above November in **six of seven years**:

| | 2019 | 2020 | 2021 | 2022 | 2023 | 2024 | 2025 |
|---|---|---|---|---|---|---|---|
| Dec − Nov (pp) | +5.75 | −0.07 | +0.11 | +0.53 | **+0.90** | +0.50 | +0.39 |

Since 2021 the December step grows year on year. December 2023 is the largest of
the recent ones, not an isolated event. The one negative year, 2020, sits inside
the collapse the bank line went through that year — the very stretch section 4
now flags as unexplained.

**Check 3 — spread across the book, not one product.** Every modality moves up.
Weighting each by its share of the December book reproduces the observed step
almost exactly:

| Modality | Move | Weight | Contribution |
|---|---|---|---|
| `Empréstimos` | +0.80 pp | 40.2% | +0.32 pp |
| `Outros créditos` | +1.97 pp | 11.7% | +0.23 pp |
| `Financiamentos` | +1.00 pp | 18.6% | +0.19 pp |
| `Financiamentos à exportação` | +0.89 pp | 12.9% | +0.12 pp |
| remaining six | — | 16.6% | +0.04 pp |
| **total** | | | **+0.90 pp** (observed +0.89) |

No single product carries the move. `Outros créditos` has the largest percentage
swing but only an eighth of the book; `Empréstimos` moves least among the top four
and contributes most, because it *is* the book.

**Where that leaves the peak.** Not arithmetic. Seasonal, with the pattern present
in six of seven years and strengthening since 2021. Book-wide rather than
product-specific. Together those point at a year-end registration effect rather
than a change in what is being lent — and the stock nature of the series makes a
30% one-month jump hard to attribute to new lending alone, which points further
towards re-registration of existing contracts.

**That last step is a hypothesis, not a result.** The series cannot separate new
credit from re-registered credit, and nothing here identifies a cause.

**What none of these can do.** None of them names a cause. They bound the space:
they can rule out an arithmetic artefact, rule the seasonal story in or out, and
say whether the movement is a book-wide practice or one product. Naming the cause
would need each institution's credit policy, which is not public. The useful
sentence in an interview is not *"I don't know"* — it is *"I ruled out the
denominator, tested seasonality across seven years, and localised the movement;
past that point the data stops and I would need the lenders' own guidance."*

## 5. Diagnosis: what makes a proportion unstable

The cut-off criterion went through two versions. Both are recorded here, because
the reason for each change is part of the method.

**First attempt — a floor on market share.** The natural reflex when facing an
unstable line is to drop small segments: *"I only analyse whoever holds at least
1% of the market."* Discarded before it reached production, for two reasons.

The first is fatal on its own: by the section 2 table, `Fintech` is **smaller**
than `Instituição de pagamento`. Any floor on relative size that removes the
unstable line removes the line carrying the thesis as well. The criterion would
cut out the object of study.

The second is more fundamental. Market share and stability of a proportion are
not the same property. A small segment with many small contracts produces a
stable proportion; a larger segment with very few contracts produces an unstable
one. What needs measuring is **stability**, so the criterion has to aim at it
directly.

**Second attempt — a stability criterion.** The cut targets fragility itself: how
much a single average-sized contract can shift the proportion in a given month.
Where that displacement is large, the line is not trustworthy, regardless of how
big the segment is.

In [ ]:
# `contracts` is withheld whenever a cell holds few operations.
# Dropping those rows understates the total — and understates it MORE in small
# segments, which are exactly the ones this thesis observes. The filter would be
# biased against fintechs for the wrong reason.
#
# Instead of dropping: each masked cell counts as at least one contract.

known = base['numero_de_operacoes'].where(~base['operacoes_suprimidas'], 0)

diag = (base.assign(_known=known)
            .groupby(['data_base', 'segmento'], observed=True)
            .agg(outstanding=('carteira_ativa', 'sum'),
                 known_contracts=('_known', 'sum'),
                 masked_cells=('operacoes_suprimidas', 'sum'),
                 cells=('operacoes_suprimidas', 'size')))

diag['min_contracts'] = diag['known_contracts'] + diag['masked_cells']
diag['masked_share'] = diag['masked_cells'] / diag['cells']

# Displacement caused by one average-sized contract, in percentage points.
#
# It is a CEILING with respect to the arithmetic: min_contracts is a floor on the
# contract count, so 100 / min_contracts is the largest value this ratio can take.
# It is a FLOOR with respect to reality: value is more concentrated than count, so
# one large contract shifts the proportion by more than 1/N. The metric bounds the
# average case, not the worst case.
diag['max_shift_pp'] = 100 / diag['min_contracts']

diag.groupby('segmento', observed=True)[
    ['max_shift_pp', 'masked_share']].describe().round(3)

In [ ]:
shift = diag['max_shift_pp'].unstack('segmento')

CEILING_PP = 0.1   # see DECISIONS.md — equivalent to requiring ~1,000 contracts

fig, ax = style.figure('Maximum shift caused by one average-sized contract',
                       'percentage points',
                       subtitle='Business credit, Brazil — log scale; the higher the line, the more fragile the proportion')
shift.plot(ax=ax, logy=True, color=style.colors(shift.columns))
style.dash(ax)
ax.axhline(CEILING_PP, color=style.YELLOW,
           linestyle='--', linewidth=1.2, alpha=0.9)
ax.text(shift.index[0], CEILING_PP * 1.08, ' ceiling = 0.1 pp (~1,000 contracts)',
        color=style.YELLOW, fontsize=8.5, va='bottom')
style.finalize(ax, 'Source: SCR.data, Central Bank of Brazil (ODbL)')
plt.show()

### Reading

**`Banco`.** Present in all 90 months, and the displacement rounds to 0.000 pp at
three decimals in every one of them — under 0.0005 pp, at least two hundred times
below the ceiling. That earns its role as a control group through a route
independent of how the line looks: whatever moves the bank series, it is not a
handful of contracts. Note that this says nothing about the series being *stable*;
section 4 shows it is not. Fragility and movement are different properties, and
this criterion only measures the first.

**`Fintech`.** Present in 84 of the 90 months — the segment does not appear in
SCR.data until July 2019. Median displacement 0.002 pp, but a maximum of 0.781 pp,
nearly eight times the ceiling. The spread is the point: early months rest on so
few contracts that one of them carries a large share of the proportion, and those
are the months the filter removes.

**`Instituição de pagamento`.** Present in 78 months and by far the noisiest.
Median 0.000 pp, mean 2.994 pp, maximum **25.0 pp** — one month in which a single
average contract could move a quarter of the proportion. A median near zero beside
a mean of three is the signature of a segment that is usually fine and
occasionally catastrophic, which is exactly why a criterion applied month by month
beats any judgement applied to the segment as a whole.

**The suppression rates are their own finding.** Share of cells with the contract
count withheld: `Banco` 31.5%, `Instituição de pagamento` 50.3%, `Fintech` **75.3%**
— reaching 97.1% in one month, and 100% for one payment-institution month. Three
out of four fintech cells hold too few operations to be disclosed. This is why the
imputation choice in the cell above is not a technicality: for fintechs it governs
three quarters of the data.

**A caveat about the metric itself.** `100/N` is the displacement of an
*average-sized* contract. Real contracts vary enormously, and value is far more
concentrated than count, so a large contract moves the proportion by more than
`100/N` suggests. The metric is therefore a **floor** on the real displacement,
not a measurement of it. It serves to rank months from more to less trustworthy;
it does not license the sentence *"this month carries an error of X points"*. An
imperfect metric used under the right label is honest work. The same metric
presented as precise would be the opposite.

The metric is doing what it was built to do: it flags fragility, and it flags it
without knowing anything about the thesis. Note that a jump in this metric means
a **drop in contract count**, which is why it doubles as a break detector — it
located discontinuities independently of section 1.1.

In [ ]:
eligible = shift <= CEILING_PP
filtered = series.where(eligible[series.columns])

fig, ax = style.figure('Unavailable borrower size as a share of outstanding credit',
                       'Share of outstanding',
                       subtitle='Business credit, Brazil — months passing the stability criterion only')
# The raw series is drawn faintly underneath so the months the criterion removed
# stay visible. Without it, a rejected month and a missing month look identical,
# and the distinction between the two is the argument of this section.
series.plot(ax=ax, color=style.colors(
    series.columns), alpha=0.18, legend=False)
filtered.plot(ax=ax, color=style.colors(filtered.columns))
style.dash(ax)
ax.set_ylim(0, YMAX)   # same ruler as the unfiltered chart above
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1, decimals=0))
style.mark(ax, pd.Timestamp('2025-07-31'),
           'Jul 2025 — unexplained movement (see DECISIONS.md)')
style.finalize(ax, 'Source: SCR.data, Central Bank of Brazil (ODbL)')
plt.show()

print('months kept per segment (out of', len(series), '):')
print(eligible[series.columns].sum())

### What the filter fixed, and what it did not

Months passing the criterion, nationwide: **`Banco` 90/90, `Fintech` 71/90,
`Instituição de pagamento` 64/90.**

**Fixed for fintechs.** The early stretch drops out, and what remains still climbs
from 5.0% (Aug 2020) to 40.3% (Jun 2026) — using only months that passed. The
thesis survives its own filter and comes out stronger: the rise no longer depends
on months with a tiny denominator.

**`Banco` passes every month.** All 90. That is not a courtesy of the criterion;
it is a consequence of the segment holding on the order of a million contracts a
month. The control group is admitted by the same rule that excludes a fifth of
the fintech series.

> **Amendment — payment institutions.** The Paraná version declared this segment
> **not analysable in value terms** and excluded it. Nationwide, 64 of 90 months
> pass the criterion, so the exclusion no longer follows from the data and is
> withdrawn.
>
> **What replaces it is a different problem.** The national curve — 0.1% (Mar
> 2021) → 67.3% (Dec 2023) → 8.0% (Jun 2026) — is not a trend. It is a spike and
> a collapse, and it happens in months the stability criterion approves. Passing
> the criterion means "few contracts cannot explain this"; it does not mean the
> movement is economic. A rise and fall of sixty points in a segment holding 0.21%
> of the market most plausibly reflects the entry or exit of a small number of
> reporting institutions, not a change in underwriting method.
>
> The segment is therefore **kept in the charts and excluded from the thesis**,
> which is a narrower claim than before and rests on a stated reason rather than
> on a filter that did not reach it. Establishing what actually happened there is
> a separate investigation, listed in section 8.

**The unit mismatch is still real.** The criterion is denominated in contract
counts while the series is denominated in outstanding value. Where value is
concentrated in a few large contracts, a proportion in BRL can stay fragile while
the metric reports it as safe. Lowering the ceiling does not fix this — it would
drop fintech months along the way, again for the wrong reason. A criterion
denominated in value is the proper fix and is not yet built.

## 5.1 Does the conclusion depend on the parameter?

A ceiling had to be chosen: above what displacement is a month untrustworthy
enough to leave the chart? Any number picked here is attackable — *"you chose the
cut-off that made the chart work."*

**The answer is not to defend a number. It is to show the conclusion does not
depend on it.** Everything below is recomputed under several ceilings, including
no filter at all.

In [ ]:
def reading(ceiling):
    '''Fintech series under one ceiling: extent, endpoints and slope.'''
    passing = (shift.reindex(series.index) <= ceiling).fillna(False)
    f = series.where(passing[series.columns])['Fintech'].dropna()
    if len(f) < 3:
        return None
    years = (f.index - f.index[0]).days / 365.25
    return {
        'months_kept': len(f),
        'starts': f.index[0].strftime('%Y-%m'),
        'first_%': round(f.iloc[0] * 100, 1),
        'last_%': round(f.iloc[-1] * 100, 1),
        'slope_pp_per_year': round(np.polyfit(years, f.values, 1)[0] * 100, 2),
    }


sensitivity = {f'ceiling {c}': reading(c)
               for c in [0.05, 0.10, 0.25, 0.50, 1.00]}
sensitivity['no filter'] = reading(np.inf)

pd.DataFrame(sensitivity).T

### Reading

The fintech series ends at **40.3% under every ceiling, including no filter at
all**. The slope stays between **7.05 and 8.9 percentage points per year**, always
strongly positive:

| Ceiling | Months kept | Series starts | First | Last | Slope (pp/year) |
|---|---|---|---|---|---|
| 0.05 | 69 | 2020-10 | 0.0% | 40.3% | 8.90 |
| 0.10 | 71 | 2020-08 | 5.0% | 40.3% | 8.52 |
| 0.25 | 79 | 2019-12 | 0.1% | 40.3% | 7.55 |
| 0.50 | 80 | 2019-11 | 0.0% | 40.3% | 7.45 |
| 1.00 | 84 | 2019-07 | 0.0% | 40.3% | 7.05 |
| none | 84 | 2019-07 | 0.0% | 40.3% | 7.05 |

The ceiling moves the starting point by two years and the slope by less than two
points per year. It never touches the endpoint, and no setting turns the rise into
anything but a rise.

**Why "no filter" also stops at 84 months.** It is not the filter. `Fintech` has no
rows at all in SCR.data before July 2019, so six months are absent from the series
itself and no ceiling can restore them. The identical figures in the last two rows
are the correct result, not a bug: at a ceiling of 1.00 pp every month the segment
actually has already passes.

**Why this table matters more than any defence of the chosen value.** The ceiling
decides **how much of the series is shown. It does not decide what the series
says.** That closes two objections at once:

1. *"You picked the cut-off that worked"* — there is no cut-off that makes the
   finding disappear.
2. *"Your diagnostic rests on a flawed denominator"* — true, and section 5 says
   so explicitly. But since the conclusion survives even with no diagnostic at
   all, that flaw stops being fatal.

Note what is *not* claimed here. Stability under every ceiling says the trend is
not an artefact of the filter. It says nothing about the trend being an artefact
of something else — the July 2025 source break in section 7 is untouched by this
table, because that break would move the underlying series, not the months
selected from it.

## 6. The version question, closed

The strongest available objection to this whole notebook: SCR.data is published in
two methodological versions, and if the fintech rise coincides with the migration
from version 1 to version 2, the finding is an artefact of collection rather than
market behaviour. This was carried as a **veto** — a condition that would stop
publication, not a footnote to append to it.

It is now resolved, and the resolution is negative: **the version transition
cannot explain the series.**

**The documentary argument.** Neither methodology PDF is dated, so the transition
date cannot be read off them directly. But they differ in which fields exist: the
`segmento` field and the `carteira_ativa` definition used throughout this notebook
**exist only in version 2**. The files ingested here carry both fields in every
month from January 2019 onward. A version-1 file could not have supplied them.
The Central Bank therefore republished the entire back series in version-2 format;
there is no seam in the middle of the series to align anything against.

**The structural argument.** A change in how the borrower-size field is collected
would reach every institution that reports it. It cannot arrive for fintechs and
skip banks. The bank line moving within a few percentage points while the fintech
line travels forty is not what a collection change looks like.

**What would reopen this.** Documentary evidence that version 2 changed the
*reporting obligation* specifically for smaller or newer institutions — a rule
that binds fintechs and payment institutions differently from banks. That is not
impossible, and it is the one shape of migration the argument above does not
cover. Nothing in either methodology document points to it.

## 7. July 2025 — the caveat that stays open

Section 1.1 established, from the ingestion log alone, that July 2025 is a
structural break in the composition of the published file: a 4.6% drop in total
rows against a 1.0% drop in business rows, and a permanent 1.5 pp step in the
business share.

That is a fact about the source. Whether it transmits into this series is a
separate question, and the honest answer is that it has not been tested
nationwide.

**Why it is not obviously harmless.** The metric here is a share of outstanding
value computed *within* each segment. Rows leaving the file for personal-borrower
credit cannot move it mechanically. But a change large enough to shift the file's
composition may not have been confined to personal credit, and the business
slice did lose 1% of its rows that month.

**Why it matters more than it used to.** Under the Paraná reading, the bank line
was flat for seven years and any base-wide event would have shown up in it. That
argument is gone: nationwide the bank line already moves within a few points, so
it can no longer be used as a detector fine enough to rule out a break of this
size.

**The test to run.** Compare the three months before and the three months after
July 2025, per segment, on the filtered series. If only the fintech moves, the
break does not explain the finding. If all three move together, the tail of the
series needs an explicit asterisk in the README.

In [ ]:
CUT = pd.Timestamp('2025-07-31')

before = filtered.loc[CUT -
                      pd.DateOffset(months=3):CUT - pd.DateOffset(days=1)].mean() * 100
after = filtered.loc[CUT:CUT + pd.DateOffset(months=2)].mean() * 100

comparison = (pd.DataFrame({'before_%': before, 'after_%': after})
                .assign(delta_pp=lambda t: t['after_%'] - t['before_%'])
                .round(2))
comparison

### Reading

The rule stated above was: only-fintech-moves clears the break; all-three-move
puts an asterisk on the tail. The table gives neither cleanly.

| Segment | 3 months before | 3 months after | Change |
|---|---|---|---|
| `Banco` | 3.12% | 3.44% | +0.32 pp |
| `Instituição de pagamento` | 22.61% | 26.43% | +3.83 pp |
| `Fintech` | 29.69% | 38.21% | **+8.52 pp** |

All three move, and all three move up — so the clean version of the test fails.
But they do not move together: the bank's +0.32 pp sits well inside its ordinary
month-to-month variation, while the fintech moves twenty-six times as far.

**The asterisk is warranted, and here is its size.** The filtered fintech series
runs from 5.0% to 40.3%, a rise of about 35 points. Roughly **8.5 of those points
occur across the boundary of a month with a documented structural break in the
source**. That is a quarter of the finding concentrated at a single suspect point.

**What this does and does not damage.** It does not touch the trend: truncated at
June 2025 the series still runs from 5.0% to just under 30%, and section 5.1 shows
the rise survives every ceiling. What it damages is the *headline figure*. Quoting
40.3% without the caveat means quoting a number whose final quarter of movement
coincides with a break nobody has explained.

**How the README should carry it.** State the trend on the truncated series and
give the current figure with the break marked. The honest sentence is: *"from 5% in
2020 to just under 30% by mid-2025, and 40.3% in June 2026 — with the caveat that a
structural break in the published file in July 2025 accounts for part of the final
step."*

**What would settle it.** Rebuild the fintech series from cell counts rather than
from value, for the months either side of the break. If the number of fintech cells
with no size on file rises in the same proportion as the value does, the break is
not creating the movement. That test is not built.

## 8. Next steps

- **Build a value-denominated stability criterion** — for instance, the share of a
  segment's portfolio concentrated in its largest cell that month. The
  count-based criterion cannot reach the failure mode described in section 5.
- **Establish what happened to payment institutions** between 2021 and 2026.
  The spike-and-collapse shape suggests institutions entering and leaving the
  reporting population; the ingestion log and the cell counts per month are the
  place to look.
- **Break down by state.** The nationwide extract now carries `uf`, and the Paraná
  reading of the bank line as immobile is direct evidence that geography changes
  what the series looks like. Worth quantifying rather than assuming.
- **Explain the 2019–2020 collapse in the bank line**, from 14.7% (Dec 2019) to
  about 1.5% during 2020. It is the largest single movement in the control
  segment and nothing in this notebook accounts for it.
- **Test July 2025 on counts instead of value**, as described at the end of
  section 7. It is the cheapest way to retire the largest open caveat.
- **Break down by industry:** where the no-revenue method is growing fastest.